# Power BI I: Power Query (M) and Data Modeling

Notebook for **Chapter 9** of the *analyst* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/analyst/ch09.html

Tip: change a number, predict the output, then run the cell again.

## 9.1 Power Query and M: Cleaning with a Recipe

## 9.2 Merge, Append, Parameters, Functions and Query Folding

## 9.3 Star Schema: Facts and Dimensions

## 9.4 Relationships: Cardinality, Direction, Active and Role-Playing

## 9.5 Date Table, Hierarchies, Calculated Columns vs Measures and Aggregations

## 9.6 Data Modelling Depth: Grain, SCDs, Bridges, Layers and Storage Modes

In [ ]:
import sqlite3, pandas as pd
con = sqlite3.connect(":memory:")
con.executescript("""
CREATE TABLE sales(order_id INT, cust_id INT, order_date TEXT, amount REAL);
INSERT INTO sales VALUES (1,1,'2025-01-10',100),(2,1,'2025-02-15',200),(3,1,'2025-04-05',300),
                         (4,2,'2025-02-20',150),(5,2,'2025-05-01',250);
CREATE TABLE dim1(cust_id INT, name TEXT, city TEXT);
INSERT INTO dim1 VALUES (1,'Asha','Mumbai'),(2,'Ravi','Delhi');
CREATE TABLE dim2(cust_key INT, cust_id INT, name TEXT, city TEXT, valid_from TEXT, valid_to TEXT, is_current INT);
INSERT INTO dim2 VALUES (1,1,'Asha','Pune','2024-01-01','2025-03-01',0),
                        (2,1,'Asha','Mumbai','2025-03-01','9999-12-31',1),
                        (3,2,'Ravi','Delhi','2024-01-01','9999-12-31',1);
""")
def q(sql):
    print(pd.read_sql(sql, con).to_string(index=False)); print()
print("Type 1 (overwrite):")
q("""SELECT d.city, SUM(s.amount) AS revenue FROM sales s JOIN dim1 d USING(cust_id)
     GROUP BY d.city ORDER BY d.city""")
print("Type 2 (history rows, join on the date range):")
q("""SELECT d.city, SUM(s.amount) AS revenue FROM sales s
     JOIN dim2 d ON d.cust_id = s.cust_id AND s.order_date >= d.valid_from AND s.order_date < d.valid_to
     GROUP BY d.city ORDER BY d.city""")
print("Forgot the date condition:")
q("SELECT COUNT(*) AS rows, SUM(amount) AS revenue FROM sales s JOIN dim2 d ON d.cust_id = s.cust_id")

Type 1 (overwrite):
  city  revenue
 Delhi    400.0
Mumbai    600.0

Type 2 (history rows, join on the date range):
  city  revenue
 Delhi    400.0
Mumbai    300.0
  Pune    300.0

Forgot the date condition:
 rows  revenue
    8   1600.0


In [ ]:
import sqlite3, pandas as pd
con = sqlite3.connect(":memory:")
con.executescript("""
CREATE TABLE orders(order_id INT, amount REAL);
INSERT INTO orders VALUES (1,100),(2,200),(3,300);
CREATE TABLE tags(order_id INT, tag TEXT);
INSERT INTO tags VALUES (1,'gift'),(1,'sale'),(2,'sale'),(3,'gift'),(3,'sale'),(3,'new');
""")
def q(sql):
    print(pd.read_sql(sql, con).to_string(index=False)); print()
q("SELECT SUM(amount) AS true_total FROM orders")
q("SELECT t.tag, SUM(o.amount) AS revenue FROM orders o JOIN tags t USING(order_id) GROUP BY t.tag ORDER BY t.tag")
q("SELECT SUM(o.amount) AS sum_of_tag_totals FROM orders o JOIN tags t USING(order_id)")
q("""SELECT t.tag, ROUND(SUM(o.amount * 1.0 / n.k), 1) AS weighted_revenue
     FROM orders o JOIN tags t USING(order_id)
     JOIN (SELECT order_id, COUNT(*) AS k FROM tags GROUP BY order_id) n USING(order_id)
     GROUP BY t.tag ORDER BY t.tag""")

 true_total
      600.0

 tag  revenue
gift    400.0
 new    300.0
sale    600.0

 sum_of_tag_totals
            1300.0

 tag  weighted_revenue
gift             150.0
 new             100.0
sale             350.0


In [ ]:
import sqlite3, pandas as pd
con = sqlite3.connect(":memory:")
con.executescript("""
CREATE TABLE sales(order_id INT, cust_id INT, order_date TEXT, amount REAL);
INSERT INTO sales VALUES (1,1,'2025-01-10',100),(2,1,'2025-02-15',200),(3,1,'2025-04-05',300),
                         (4,2,'2025-02-20',150),(5,2,'2025-05-01',250);
CREATE TABLE dim1(cust_id INT, name TEXT, city TEXT);
INSERT INTO dim1 VALUES (1,'Asha','Mumbai'),(2,'Ravi','Delhi');
CREATE TABLE dim2(cust_key INT, cust_id INT, name TEXT, city TEXT, valid_from TEXT, valid_to TEXT, is_current INT);
INSERT INTO dim2 VALUES (1,1,'Asha','Pune','2024-01-01','2025-03-01',0),
                        (2,1,'Asha','Mumbai','2025-03-01','9999-12-31',1),
                        (3,2,'Ravi','Delhi','2024-01-01','9999-12-31',1);
""")
def q(sql):
    print(pd.read_sql(sql, con).to_string(index=False)); print()
con.executescript("""
CREATE TABLE stg_orders(order_id INT, cust_id INT, status TEXT, amount REAL);
INSERT INTO stg_orders VALUES (1,1,'delivered',100),(2,2,'returned',50),(2,2,'returned',50),
                              (3,9,'shipped',70),(4,1,NULL,20),(5,1,'delivered',-5);
""")
tests = {
 "unique order_id":        "SELECT COUNT(*) FROM (SELECT order_id FROM stg_orders GROUP BY order_id HAVING COUNT(*) > 1)",
 "not_null status":        "SELECT COUNT(*) FROM stg_orders WHERE status IS NULL",
 "relationships cust_id":  "SELECT COUNT(*) FROM stg_orders o LEFT JOIN dim1 d USING(cust_id) WHERE d.cust_id IS NULL",
 "accepted_values status": "SELECT COUNT(*) FROM stg_orders WHERE status IS NOT NULL AND status NOT IN ('delivered','returned','cancelled')",
 "amount is not negative": "SELECT COUNT(*) FROM stg_orders WHERE amount < 0",
}
for name, sql in tests.items():
    failures = con.execute(sql).fetchone()[0]
    print(("FAIL " if failures else "ok   ") + name + ": " + str(failures))

FAIL unique order_id: 1
FAIL not_null status: 1
FAIL relationships cust_id: 1
FAIL accepted_values status: 1
FAIL amount is not negative: 1


## Chapter recap

## Practice set

Problem 9.1 Is `table.addcolumn` valid M?

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/analyst/ch09.html